## Data Transformation Experiments

### File Architecture

Output: data/preprocessed (train.csv/npy, test.csv/npy, validate.csv/npy), artifacts/preprocessor (preprocessor.pkl)

1. Load Dataset
2. Remove useless columns
3. Separate features and target
4. Train / Validation / Test Split
5. Identify numerical & categorical columns
6. Handle missing values
7. Categorical encoding
8. Feature scaling
9. Create preprocessing pipeline
10. Fit-transform training data
11. Transform validation/test data
12. Handle imbalance ONLY on training data
13. Save transformed arrays
14. Save preprocessor.pkl
15. Optional datatype optimization

NOTE: Create save and load functions in utils/common.py for saving and loading numpy files

## Change Directory

In [1]:
import os 
%pwd

'd:\\Automated_Predictive_Maintenance\\notebooks'

In [2]:
os.chdir('../')
%pwd

'd:\\Automated_Predictive_Maintenance'

## Load the data

In [3]:
import pandas as pd
import numpy as np

In [4]:
# load the raw data first

df = pd.read_csv("D:/Automated_Predictive_Maintenance/data/raw/predictive_maintenance.csv")
df.head(5)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [5]:
df.shape

(10000, 10)

In [6]:
df.dtypes

UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Target                       int64
Failure Type                object
dtype: object

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Target                   10000 non-null  int64  
 9   Failure Type             10000 non-null  object 
dtypes: float64(3), int64(4), object(3)
memory usage: 781.4+ KB


## Drop useless cols (UDI, Product ID, Failure Type)

In [9]:
# let's drop the useless cols

# UDI, Product ID, Failure Type
'''Failure Type and target; both are target column. I am currently taking Target cols as it is a binary class calssification,
but using Faliure Type as feature will lead to data leakage. hence, best option right now is to drop the Failure Type column'''

df = df.drop(columns=['UDI', 'Product ID', 'Failure Type'])


In [10]:
df.head(5)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Type                     10000 non-null  object 
 1   Air temperature [K]      10000 non-null  float64
 2   Process temperature [K]  10000 non-null  float64
 3   Rotational speed [rpm]   10000 non-null  int64  
 4   Torque [Nm]              10000 non-null  float64
 5   Tool wear [min]          10000 non-null  int64  
 6   Target                   10000 non-null  int64  
dtypes: float64(3), int64(3), object(1)
memory usage: 547.0+ KB


## Seperate Feature and Target

In [12]:
X = df.drop(columns=['Target'])
y = df['Target']

In [13]:
X.shape

(10000, 6)

In [14]:
y.shape

(10000,)

## Train / Validation / Test Split

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

In [16]:
print(f"Original X shape: {X.shape}")
print("---------------------------------")
print(f"X_train shape:   {X_train.shape}  (~70%)")
print(f"X_valid shape:   {X_valid.shape}  (~15%)")
print(f"X_test shape:    {X_test.shape}  (~15%)")

Original X shape: (10000, 6)
---------------------------------
X_train shape:   (7000, 6)  (~70%)
X_valid shape:   (1500, 6)  (~15%)
X_test shape:    (1500, 6)  (~15%)


In [17]:
print(f"Original X shape: {y.shape}")
print("---------------------------------")
print(f"y_train shape:   {y_train.shape}  (~70%)")
print(f"y_valid shape:   {y_valid.shape}  (~15%)")
print(f"y_test shape:    {y_test.shape}  (~15%)")

Original X shape: (10000,)
---------------------------------
y_train shape:   (7000,)  (~70%)
y_valid shape:   (1500,)  (~15%)
y_test shape:    (1500,)  (~15%)


### checking the stratify job

In [18]:
print("Original y distribution:")
print(pd.Series(y).value_counts(normalize=True))
print("\nTrain y distribution:")
print(pd.Series(y_train).value_counts(normalize=True))
print("\nValidation y distribution:")
print(pd.Series(y_valid).value_counts(normalize=True))
print("\nTest y distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

Original y distribution:
Target
0    0.9661
1    0.0339
Name: proportion, dtype: float64

Train y distribution:
Target
0    0.966143
1    0.033857
Name: proportion, dtype: float64

Validation y distribution:
Target
0    0.966
1    0.034
Name: proportion, dtype: float64

Test y distribution:
Target
0    0.966
1    0.034
Name: proportion, dtype: float64


## Identify Numerical and Categorical Columns

In [19]:
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_features = X_train.select_dtypes(include=['object', 'category']).columns

In [20]:
num_features

Index(['Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'],
      dtype='object')

In [21]:
cat_features

Index(['Type'], dtype='object')

### NULL VALUES

In [22]:
# do we have any null value? Let's check
X_train.isnull().sum()

Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
dtype: int64

In [23]:
X_test.isnull().sum()

Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
dtype: int64

In [24]:
df.isnull().sum()

Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Target                     0
dtype: int64

### we dont have any null value in entire dataset

### num_features need scaling, cat_features need encoding

## Categorical Encoding

cat_features 'Type' represent Ordinal data so, here we use label encoding to maintain the relation between L for LOW, M for MEDIUM, H for High

In [25]:
X_train['Type'].value_counts()

Type
L    4217
M    2078
H     705
Name: count, dtype: int64

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# 2. Group them together inside a ColumnTransformer (The Blueprint)
ordinal_mapping = [['L', 'M', 'H']]

preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', OrdinalEncoder(categories=ordinal_mapping), cat_features),
        ('numeric', StandardScaler(), num_features)
    ]
)

# 3. Execute Phase 3: Fit and Transform your sets
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_valid_preprocessed = preprocessor.transform(X_valid)
X_test_preprocessed = preprocessor.transform(X_test)

### SMOTE

In [27]:
from imblearn.over_sampling import SMOTE
oversample = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = oversample.fit_resample(X_train_preprocessed, y_train)

print("Final Training Shape (Balanced):", X_train_resampled.shape)
print("Final Validation Shape:", X_valid_preprocessed.shape)
print("Before SMOTE:", pd.Series(y_train).value_counts())
print("After SMOTE:", pd.Series(y_train_resampled).value_counts())

Final Training Shape (Balanced): (13526, 6)
Final Validation Shape: (1500, 6)
Before SMOTE: Target
0    6763
1     237
Name: count, dtype: int64
After SMOTE: Target
0    6763
1    6763
Name: count, dtype: int64


In [28]:
import joblib

### One last time, Checking the shape of data to verify the changes

In [29]:
print(X_train_resampled.shape)
print(y_train_resampled.shape)
print(X_valid_preprocessed.shape)
print(y_valid.shape)
print(X_test_preprocessed.shape)
print(y_test.shape)

(13526, 6)
(13526,)
(1500, 6)
(1500,)
(1500, 6)
(1500,)


### Saving all the data (X_train, X_valid, X_test, y_train, y_valid, y_test)

In [30]:
os.makedirs("data/preprocessed", exist_ok=True)
os.makedirs("artifacts/preprocessors", exist_ok=True)

# Save the processed numpy arrays for modeling
np.save("data/preprocessed/X_train.npy", X_train_resampled)
np.save("data/preprocessed/y_train.npy", y_train_resampled.to_numpy())
np.save("data/preprocessed/X_valid.npy", X_valid_preprocessed)
np.save("data/preprocessed/y_valid.npy", y_valid.to_numpy())
np.save("data/preprocessed/X_test.npy", X_test_preprocessed)
np.save("data/preprocessed/y_test.npy", y_test.to_numpy())

# Save the fitted preprocessor so your production API can use it to transform raw inputs!
joblib.dump(preprocessor, "artifacts/preprocessors/preprocessor.pkl")
print("All artifacts successfully saved!")

All artifacts successfully saved!
